# Занятие 1: Исследовательские вопросы, данные и шкалы измерения
### Практический кейс: Что на самом деле измеряет Индекс развития человеческого потенциала?
*Курс: Байесовский анализ эмпирических данных (2026)*

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/01_research_questions_data_measurement_ru.ipynb)
*English version: [01_research_questions_data_measurement.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/01_research_questions_data_measurement.ipynb)*

В этой лабораторной работе мы разберем принципиальное различие между абстрактными научными конструктами и наблюдаемыми эмпирическими индикаторами на примере Индекса развития человеческого потенциала (ИРЧП) по штатам США. Мы изучим доминирование дисперсии, разгадаем парадокс Кентукки против Южной Каролины и построим калибровку рангов.


In [ ]:
import sys, os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# Auto-download dataset if not found locally
DATA_PATH = "data/hdi_state.csv"
if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    url = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/hdi_state.csv"
    urllib.request.urlretrieve(url, DATA_PATH)

df = pd.read_csv(DATA_PATH)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print(f"Dataset loaded: {len(df)} rows, columns: {list(df.columns)}")


## 1. Составное измерение и доминирование индикаторов

### Идея
Индекс развития человеческого потенциала объединяет индикаторы здоровья, образования и дохода. Номинально каждое измерение входит в формулу с равным весом (1/3). На практике личный доход между штатами варьирует значительно сильнее, чем продолжительность жизни или грамотность, поэтому доход начинает численно доминировать над остальными измерениями. Мы рассчитываем ранги штатов по доходу и ИРЧП, чтобы проверить, насколько композитный индекс дублирует сырой доход.


In [ ]:
# Filter 50 states with observed personal income in 2000
valid = df.dropna(subset=["income_2000"]).copy()
valid["rank_income"] = valid["income_2000"].rank(ascending=False)
valid["rank_hdi"] = valid["hdi"].rank(ascending=False)
valid["rank_gain"] = valid["rank_income"] - valid["rank_hdi"]

r_p, p_p = pearsonr(valid["income_2000"], valid["hdi"])
r_s, p_s = spearmanr(valid["rank_income"], valid["rank_hdi"])

print(f"Pearson Correlation (Raw Scale):   r   = {r_p:.3f} (p = {p_p:.2e})")
print(f"Spearman Correlation (Rank Scale): r_s = {r_s:.3f} (p = {p_s:.2e})")
print("\nTop 5 States by HDI:")
print(valid.sort_values("rank_hdi")[["state", "income_2000", "rank_income", "hdi", "rank_hdi", "rank_gain"]].head())


### На что обратить внимание
Корреляция между доходом и ИРЧП высока ($r = 0{,}71, r_s = 0{,}86$). Это подтверждает, что экономическое благосостояние сильно коррелирует с индексом. Тем не менее, доход объясняет далеко не всё: некоторые штаты достигают значительно более высоких социальных результатов, чем предсказывает уровень их доходов.


## 2. Расхождения в измерениях: парадокс Кентукки и Южной Каролины

### Идея
Гельман, Хилл и Вехтари (*Regression and Other Stories*, гл. 2) обратили внимание на классическое расхождение измерений между Южной Каролиной и Кентукки. В 2000 году подушевой доход в этих двух штатах был практически одинаков (разница составляла всего $11 в год), однако их показатели ИРЧП резко разошлись.


In [ ]:
sc_ky = valid[valid["state_abb"].isin(["SC", "KY"])][["state", "income_2000", "rank_income", "hdi", "rank_hdi"]]
print(sc_ky.to_string(index=False))

diff_income = abs(sc_ky.iloc[0]["income_2000"] - sc_ky.iloc[1]["income_2000"])
diff_hdi = abs(sc_ky.iloc[0]["hdi"] - sc_ky.iloc[1]["hdi"])
national_range = valid["hdi"].max() - valid["hdi"].min()

print(f"\nIncome difference: ${diff_income:.2f} (Rank #{sc_ky.iloc[0]['rank_income']:.0f} vs #{sc_ky.iloc[1]['rank_income']:.0f})")
print(f"HDI difference:    {diff_hdi:.3f} (Rank #{sc_ky.iloc[0]['rank_hdi']:.0f} vs #{sc_ky.iloc[1]['rank_hdi']:.0f})")
print(f"Share of national HDI range: {diff_hdi / national_range * 100:.1f}%")


### На что обратить внимание
При разнице в доходах всего в $11 Кентукки и Южная Каролина различаются на 13 позиций в национальном рейтинге ИРЧП, покрывая почти треть ($31{,}3\%$) всего национального диапазона! Это эмпирически доказывает, что один лишь доход не может служить полным заменителем конструкта «человеческое развитие».


## 3. Калибровка рангов и социальная эффективность

### Идея
График калибровки ранг-ранг сопоставляет ранг штата по доходу с его рангом по ИРЧП. Диагональ $y = x$ обозначает идеальный паритет: штат, развитие которого точно соответствует его доходам, лежит строго на диагонали. Штаты выше диагонали достигают более высокого уровня развития, чем предсказывает их богатство (социальная эффективность), а штаты ниже диагонали отстают.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
scatter = ax.scatter(valid["rank_income"], valid["rank_hdi"], c=valid["rank_gain"], cmap="viridis", s=55, edgecolor="black", alpha=0.85)
ax.plot([1, 50], [1, 50], color="#e07a5f", linestyle="--", lw=2, label="Equal Rank (y = x)")

# Annotate key diagnostic states
key_states = ["SC", "KY", "UT", "CT", "MS"]
for _, row in valid[valid["state_abb"].isin(key_states)].iterrows():
    ax.annotate(row["state_abb"], (row["rank_income"], row["rank_hdi"]),
                textcoords="offset points", xytext=(5, 5), fontweight="bold", fontsize=9)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Rank Gain (Income Rank - HDI Rank)")
ax.set_xlabel("Income Rank (1 = Highest Income)")
ax.set_ylabel("HDI Rank (1 = Highest HDI)")
ax.set_title(f"Rank-Rank Calibration Across 50 U.S. States (Spearman r_s = {r_s:.2f})")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


### На что обратить внимание
Юта (UT) представляет собой яркий пример: занимая лишь 38-е место по личным доходам, штат занимает 24-е место по ИРЧП ($+14$ позиций выигрыша), демонстрируя высокую эффективность конвертации ресурсов в качество жизни. Напротив, штаты с высоким доходом, но отстающей социальной инфраструктурой, не реализуют свой экономический потенциал.
